
# Section 5 - Sampler

## Certification weight: 12%

## Objectives 
- Set sampler primitive options such as dynamical decoupling
- Understand the theoretical background behind the sampler primitive 



## The Sampler primitive
The Sampler primitive executes one or more quantum circuits and returns the measurement outcome probabilities (or counts) obtained from repeated shots. It is the primitive used when you care about which bitstrings are produced by a circuit and how often they occur.  
Using the coin-toss analogy, the Sampler tosses the coin many times and records the outcome of each shot, allowing us to estimate the probability distribution of the results.

In Qiskit 2.0, the sampler is called `SamplerV2` and is part of the `qiskit_ibm_runtime` package


# Running the Sampler

the Primitive Unified Bloc (PUB) is a tuple sent to `estimator.run()` or `sampler.run()` to map a circuit with the associed data (number of shots, parameters, ...)

For a sampler, the PUB looks like: 
`(<transpiled circuit including measurements>, <parameter values (optional array)>, <number of shots (optional)> )`


In [2]:
# Get a real backend
from qiskit_ibm_runtime import QiskitRuntimeService

#Load saved account
service = QiskitRuntimeService()

# Select a backend
backend = service.least_busy(operational=True, simulator=False)

In [3]:
# Create a bell-state circuit
from qiskit import QuantumCircuit, transpile

bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0,1)
bell.measure_all()

# transpile
bell_transpiled = transpile(bell, backend)

# Construct PUB 
bell_pub = (bell_transpiled, )     # Tuple including  a the transpilled cicuit, no parameters, and no shots  (so, using the default values)

In [4]:
from qiskit_ibm_runtime import SamplerV2

# Initialize the sampler
sampler = SamplerV2(backend)

# Run the circuit on a backend
bell_job = sampler.run([bell_pub], shots = 15)

# Extract results
bell_result = bell_job.result()


## Getting the data from the Sampler

`job.result()` returns a PrimitiveResult, containing a `SamplerPubResult` per PUB, and some metadata:
- the shape: the number of parameters in the PUB
- the number of shots
- the number of bits
- the date-time of the execution

We can get the metadata using `.metadata`

In [5]:
# PubResult : data containing the pub's execution results
bell_result

PrimitiveResult([SamplerPubResult(data=DataBin(meas=BitArray(<shape=(), num_shots=15, num_bits=2>)), metadata={'circuit_metadata': {}})], metadata={'execution': {'execution_spans': ExecutionSpans([DoubleSliceSpan(<start='2026-05-30 14:32:52', stop='2026-05-30 14:32:56', size=15>)])}, 'version': 2})

In [7]:
# PubResultMetadata : results metadata
bell_result.metadata

{'execution': {'execution_spans': ExecutionSpans([DoubleSliceSpan(<start='2026-05-30 14:32:52', stop='2026-05-30 14:32:56', size=15>)])},
 'version': 2}

As the PrimitiveResult contains the data from each PUBs, the data is stored in a table. The index 0 is the first PUB, the index 1 is the second, and so on...  
We can get the data of each circuit using `PrimitiveResult[<index>].data`  


In [8]:
bell_result_data = bell_result[0].data
bell_result_data

DataBin(meas=BitArray(<shape=(), num_shots=15, num_bits=2>))

The measurement data is an attribute of the `DataBin` data structure. Its name is the name of the classical register (defined manually when creating the classical register, or named `meas` if the classical register had been created using `measure_all()`)

In [9]:
#Accessing BitArray from the DataBin
bits = bell_result_data.meas
bits

BitArray(<shape=(), num_shots=15, num_bits=2>)

The measurement result has some attributes:
- The shape: the numbers of parameters into the PUB (empt if no parameter had been provided)
- the num_bits: the number of classical register
- the num_shots: the number of shots per parameter
- the array: a numpy compatible array. IT contain the decimal for of the output: 0-> 00  3->11

In [10]:
print("Shape:", bits.shape)  
print("num_bits", bits.num_bits) 
print("num_shots", bits.num_shots)
print("array:", bits.array)

Shape: ()
num_bits 2
num_shots 15
array: [[3]
 [0]
 [3]
 [3]
 [0]
 [3]
 [3]
 [0]
 [3]
 [0]
 [3]
 [3]
 [0]
 [3]
 [0]]


We can get the bitstring (the raw data), the counts (the number of time each result happened) or the counts in decimal format

In [11]:
print("bitstring:      ", bits.get_bitstrings())
print("get_counts:     ", bits.get_counts())
print("get_int_counts: ", bits.get_int_counts())

bitstring:       ['11', '00', '11', '11', '00', '11', '11', '00', '11', '00', '11', '11', '00', '11', '00']
get_counts:      {'11': 9, '00': 6}
get_int_counts:  {3: 9, 0: 6}


To sum up the data structure of a Sampler result, here is the hierarchy  that can be found in https://quantum.cloud.ibm.com/docs/en/guides/primitive-input-output 

```
└── PrimitiveResult
    ├── PubResult[0]
    │   ├── metadata
    │   └── data  ## In the form of a DataBin object
    │       ├── NAME_OF_CLASSICAL_REGISTER
    │       │   └── BitArray of count data for first PUB (default is 'meas')
    |       |
    │       └── NAME_OF_ANOTHER_CLASSICAL_REGISTER
    │           └── BitArray of count data (exists only if more than one
    |                 ClassicalRegister was specified in the circuit)
    ├── PubResult[1]
    |   ├── metadata
    |   └── data  ## In the form of a DataBin object
    |       └── NAME_OF_CLASSICAL_REGISTER
    |           └── BitArray of count data for second PUB
    ├── ...
    ├── ...
    └── ...
```

# Sampling a Parametrized circuit

Follow the same procedure, with a table of parameter added into the PUB.  
The result table will be multi dimentional, one colum per parameter

In [12]:
from qiskit.circuit import Parameter
import numpy as np

param = QuantumCircuit(2)
theta = Parameter("T")
param.ry(theta, 0)
param.cx(0,1)
param.measure_all()

param_transpiled = transpile(param, backend)

# let's evaluate thetas
param_values = np.linspace(0, np.pi, 10) # linspace will create a table of 10 evenly spaced values between 0 and pi 

# Construct a pub, and run it
param_pub = (param_transpiled, param_values)
param_job = sampler.run([param_pub], shots=1000)
param_results = param_job.result()
param_bits = param_results[0].data.meas

In [13]:
param_values

array([0.        , 0.34906585, 0.6981317 , 1.04719755, 1.3962634 ,
       1.74532925, 2.0943951 , 2.44346095, 2.7925268 , 3.14159265])

In [14]:
param_bits

BitArray(<shape=(10,), num_shots=1000, num_bits=2>)

In [15]:
print("Shape:", param_bits.shape)  # Number of parameters into the pub: 10 as we created a table of 10 parameters
print("num_bits", param_bits.num_bits)  # number of classical registers
print("num_shots", param_bits.num_shots) # number of shots per parametrization
print("array:", param_bits.array) # numpy compatible array. 0-> 00  3->11

Shape: (10,)
num_bits 2
num_shots 1000
array: [[[0]
  [0]
  [0]
  ...
  [0]
  [0]
  [0]]

 [[0]
  [0]
  [0]
  ...
  [0]
  [0]
  [0]]

 [[0]
  [3]
  [0]
  ...
  [0]
  [0]
  [0]]

 ...

 [[3]
  [3]
  [3]
  ...
  [3]
  [3]
  [3]]

 [[2]
  [3]
  [3]
  ...
  [3]
  [3]
  [3]]

 [[3]
  [3]
  [3]
  ...
  [3]
  [3]
  [3]]]


When using `get_counts()` without parameters when sampling a parametrized circuit, all the counts will be displayed. In this case, 1000shots*10 parameters = 10000 values  
To get the result of a specific parameter, the experiment index has to be set as the `get_counts()` argument:

In [16]:
param_bits.get_counts()

{'00': 4868, '11': 4844, '10': 181, '01': 107}

In [17]:
for i, theta_val in enumerate(param_values):
    counts_i = param_bits.get_counts(i)
    print(f"Theta = {theta_val:.3f} → counts = {counts_i}")

Theta = 0.000 → counts = {'00': 962, '11': 8, '10': 27, '01': 3}
Theta = 0.349 → counts = {'00': 956, '10': 12, '11': 32}
Theta = 0.698 → counts = {'00': 856, '11': 120, '10': 21, '01': 3}
Theta = 1.047 → counts = {'00': 736, '11': 243, '10': 21}
Theta = 1.396 → counts = {'10': 24, '00': 562, '11': 405, '01': 9}
Theta = 1.745 → counts = {'11': 578, '00': 387, '10': 21, '01': 14}
Theta = 2.094 → counts = {'00': 253, '11': 714, '01': 16, '10': 17}
Theta = 2.443 → counts = {'11': 848, '10': 17, '00': 116, '01': 19}
Theta = 2.793 → counts = {'10': 14, '11': 937, '01': 19, '00': 30}
Theta = 3.142 → counts = {'11': 959, '01': 24, '10': 7, '00': 10}


## Changing Sampler parameters

When the sampler is instancied, it's possible to configure its options:
- sampler.options.default_shots = 512   # default: 4096
- sampler.options. ...

Or update in one command all some options

```
sampler.options.update({    
       'default_shots': 512,
       '...':
  })
```

Or create an option object, and pass it as an argument to the sampler when creating it:

```
from qiskit_ibm_runtime.options import SamplerOptions
options = SamplerOptions()
options.default_shots = 2048
sampler = Sampler(mode=backend, options=options)
```

## Error mitigation and suppression techniques

https://quantum.cloud.ibm.com/docs/en/guides/error-mitigation-and-suppression-techniques

**Twirling** consists in adding gates or measurements that have no effect on the final measure, in order to limit the noise



Enable twirling using the following options:   
sampler.options.twirling.enable_gates = True (false by default)  
sampler.options.twirling.enable_measure = True (false by default)  

https://quantum.cloud.ibm.com/docs/en/api/qiskit-ibm-runtime/options-twirling-options

other options: 
- num_randomizations (The number of circuit instances to draw from the ensemble of twirled circuits)
- shots_per_randomization (The number of shots to sample from each circuit instance)
- strategy (active, active-circuit, active-accum, all / Default: active-accum)

**Dynamic Decoupling** 
Insert pulses to suppress decoherence: when no operation is sheduled in a line, it will insert pulses/gates during this interval to keep the line busy
https://quantum.cloud.ibm.com/docs/en/api/qiskit-ibm-runtime/options-dynamical-decoupling-options

Enable Dynamic Decoupling using the following options:
- sampler.options.dynamical_decoupling.enable = True  # False by default
- sampler.options.dynamical_decoupling.sequence_type = "XY4" # XY4 applies XY-X-Y ; XX applies XX, XpXm applies X-X
- sampler.options.dynamical_decoupling.scheduling_method = "alap" #asap or alat (as late as possible)

Other options: 
- skip_reset_qubits: insert dynamic decoupling for reset qubits
- extra_slack_distribution: 